# 3 · Ensembles & Interpretability

*[ensemble-codesim](https://github.com/jorge-martinez-gil/ensemble-codesim) tutorial series — the core idea of the paper.*

Notebook 02 showed that individual measures are **complementary**: they fail on different inputs. Here we:

1. turn similarity scores into **decisions** and score them properly (precision / recall / F1 / MCC / ROC-AUC);
2. evaluate on the repository's **real, committed benchmark data** (nothing is fabricated — we just read the feature files produced by the 21 production measures);
3. compare **single measures** against **transparent** (mean / vote) and **learned** (logistic-regression) ensembles;
4. **explain** a single ensemble decision.

> Reproducibility note: the files in `outputs/` were generated by running `exec-bigclonebench.py` / `exec-karnalim.py`, which apply all 21 measures to every pair and store one row per pair: `[m1, m2, ..., m21, label]`. We load those numbers and compute metrics on them. You can regenerate the files yourself.

In [ ]:
# --- setup: make the tutorial modules importable from anywhere in the repo ---
import os, sys
def _find_tutorials(start):
    d = os.path.abspath(start)
    for _ in range(8):
        if os.path.exists(os.path.join(d, "codesim_edu.py")):
            return d
        if os.path.exists(os.path.join(d, "tutorials", "codesim_edu.py")):
            return os.path.join(d, "tutorials")
        nd = os.path.dirname(d)
        if nd == d: break
        d = nd
    raise RuntimeError("Run this notebook from inside the ensemble-codesim repo.")
TUT  = _find_tutorials(os.getcwd())
REPO = os.path.dirname(TUT)
DATA = os.path.join(REPO, "outputs")
sys.path.insert(0, TUT); sys.path.insert(0, os.path.join(TUT, "examples"))
print("tutorials:", TUT); print("data dir :", DATA)

## From scores to decisions

A measure gives a number; a **detector** needs a yes/no. We pick a **threshold** `t` and predict "clone" when `score >= t`. Different thresholds trade **precision** (of the pairs we flagged, how many were true clones) against **recall** (of the true clones, how many we caught). Summary metrics:

- **F1** — harmonic mean of precision & recall (our headline; robust to imbalance-ish).
- **MCC** — Matthews correlation; balanced even under heavy class imbalance.
- **ROC-AUC** — threshold-free ranking quality.

`eval_edu` implements these in pure NumPy.

In [ ]:
import numpy as np
import eval_edu as E

# toy sanity check
s = [0.1, 0.4, 0.35, 0.8, 0.6]
y = [0,   0,    1,    1,   0]
print("ROC-AUC      :", round(E.roc_auc(s, y), 3))
best = E.best_threshold_f1(s, y)
print("best-F1 thr  :", round(best["threshold"], 3), "-> F1 =", round(best["f1"], 3),
      "precision =", round(best["precision"], 3), "recall =", round(best["recall"], 3))

## Real data: per-measure performance on Karnalim (IR-Plag)

The Karnalim feature file is small (a few hundred pairs) and runs instantly. Let us rank the 21 production measures by their best-F1 and by ROC-AUC.

In [ ]:
path = os.path.join(DATA, "output-karnalim.txt")
X, y, names = E.load_feature_file(path)
print(f"Karnalim: {X.shape[0]} pairs, {X.shape[1]} measures, "
      f"positives = {int(y.sum())} ({y.mean():.0%} of pairs)\n")

rows = []
for i, nm in enumerate(names):
    auc = E.roc_auc(X[:, i], y)
    f1 = E.best_threshold_f1(X[:, i], y)["f1"]
    rows.append((nm, auc, f1))
rows.sort(key=lambda r: -r[2])
print(f"{'measure':10s} {'ROC-AUC':>8s} {'best-F1':>8s}")
print("-" * 28)
for nm, auc, f1 in rows:
    print(f"{nm:10s} {auc:8.3f} {f1:8.3f}")
best_single = max(r[2] for r in rows)
print(f"\nBest single-measure F1 (oracle threshold): {best_single:.3f}")

## Two transparent ensembles

Before any machine learning, two fully interpretable combinations:

- **Mean ensemble** — min-max normalise each measure, then average. One knob, no training.
- **Vote ensemble** — each measure votes "clone" if it exceeds its own best-F1 threshold; the score is the fraction voting yes.

These need no labels to *combine* (only to pick a final threshold), so they are essentially unsupervised.

In [ ]:
me = E.mean_ensemble(X)
thr = np.array([E.best_threshold_f1(X[:, i], y)["threshold"] for i in range(X.shape[1])])
ve = E.vote_ensemble(X, thr)

for label_, sc in [("mean ensemble", me), ("vote ensemble", ve)]:
    b = E.best_threshold_f1(sc, y)
    print(f"{label_:14s} F1 = {b['f1']:.3f}   ROC-AUC = {E.roc_auc(sc, y):.3f}")
print(f"{'best single':14s} F1 = {best_single:.3f}")
print("\nNote: on this small, ~77%-positive dataset the best *single* measure is")
print("already very strong, so a naive average need not beat it. Read on.")

## A learned ensemble — and an honest evaluation

The paper combines the 21 measures with a **trained** model (Random Forest / XGBoost). Here we use the simplest learned combiner — a **logistic regression** over the 21 measures, in pure NumPy — and evaluate it **properly** with a train/test split repeated over several seeds (so we report mean ± std, not a single lucky number).

We compare three systems on held-out data:
- **best single measure** (chosen on the *training* split, evaluated on test),
- **mean ensemble**,
- **logistic-regression ensemble**.

In [ ]:
def evaluate(X, y, n_splits=5, test_frac=0.3):
    res = {"best_single": [], "mean": [], "logreg": []}
    for seed in range(n_splits):
        tr, te = E.train_test_split_idx(len(y), test_frac, seed)
        Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]
        # best single: select measure+threshold on train, score on test
        bf, bt = -1.0, None
        for i in range(X.shape[1]):
            m = E.best_threshold_f1(Xtr[:, i], ytr)
            if m["f1"] > bf:
                bf = m["f1"]; bt = (i, m["threshold"])
        res["best_single"].append(E.metrics_at(Xte[:, bt[0]], yte, bt[1])["f1"])
        # mean ensemble (scale by train stats)
        lo, hi = Xtr.min(0), Xtr.max(0); span = np.where(hi > lo, hi - lo, 1.0)
        mtr, mte = ((Xtr - lo) / span).mean(1), ((Xte - lo) / span).mean(1)
        res["mean"].append(E.metrics_at(mte, yte, E.best_threshold_f1(mtr, ytr)["threshold"])["f1"])
        # logreg ensemble
        ptr = E.logreg_fit_predict(Xtr, ytr, Xtr)
        pte = E.logreg_fit_predict(Xtr, ytr, Xte)
        res["logreg"].append(E.metrics_at(pte, yte, E.best_threshold_f1(ptr, ytr)["threshold"])["f1"])
    return res

r = evaluate(X, y)
print("Karnalim — held-out test F1 (mean +/- std over 5 splits):")
for k in ("best_single", "mean", "logreg"):
    v = np.array(r[k]); print(f"  {k:12s} {v.mean():.3f} +/- {v.std():.3f}")

## Does the conclusion hold on a different dataset?

Karnalim is small and clone-heavy. **BigCloneBench** is larger and only ~14% positive — a regime where combining measures tends to help more. (We read a subset for speed; raise `max_rows` to use more.)

In [ ]:
bcb_path = os.path.join(DATA, "output-bigclonebench.txt")
Xb, yb, _ = E.load_feature_file(bcb_path, max_rows=6000)
print(f"BigCloneBench subset: {Xb.shape[0]} pairs, positives = {int(yb.sum())} ({yb.mean():.0%})\n")
rb = evaluate(Xb, yb, n_splits=3)
print("BigCloneBench (subset) — held-out test F1 (mean +/- std over 3 splits):")
for k in ("best_single", "mean", "logreg"):
    v = np.array(rb[k]); print(f"  {k:12s} {v.mean():.3f} +/- {v.std():.3f}")

### What the numbers say (honestly)

- On **Karnalim**, the single best measure is already excellent, and a simple ensemble does **not** automatically beat it.
- On **BigCloneBench**, the **learned ensemble wins**.
- The catch: *which* single measure is best **changes from dataset to dataset** (and you do not know it in advance). The ensemble is **consistently competitive-or-better without that oracle knowledge** — that robustness is its real value.

The paper pushes this further with **non-linear** ensembles (Random Forest / XGBoost) that capture interactions a linear model cannot — see `ensembles/bagging.py` and `ensembles/boosting.py`. If you have scikit-learn installed, the next cell runs a quick Random Forest with cross-validation; otherwise it is skipped.

In [ ]:
try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score
    clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
    f1s = cross_val_score(clf, X, y, cv=5, scoring="f1")
    print(f"Random Forest 5-fold F1 on Karnalim: {f1s.mean():.3f} +/- {f1s.std():.3f}")
except Exception as e:
    print("scikit-learn not available -> skipping Random Forest demo (%s)." % type(e).__name__)
    print("Install it (pip install scikit-learn) to reproduce the paper's ensemble,")
    print("or see ensembles/bagging.py and ensembles/boosting.py.")

## Interpretability: explaining one decision

A key selling point of this approach over a black-box transformer is that **every decision is auditable**. For a single pair we can show each measure's contribution, where the measures **agree** or **disagree**, an overall **confidence**, and an **uncertainty warning** when the measures are split.

Below is a compact explainer over the transparent edu-measures (the same idea applies to all 21).

In [ ]:
from codesim_edu import all_scores

def explain_pair(code_a, code_b, threshold=0.5):
    scores = all_scores(code_a, code_b)
    names_ = list(scores); vals = np.array([scores[n] for n in names_])
    confidence = float(vals.mean()); disagreement = float(vals.std())
    verdict = "CLONE" if confidence >= threshold else "NOT A CLONE"
    order = np.argsort(-vals)
    supporting = [(names_[i], vals[i]) for i in order if vals[i] >= threshold]
    dissenting = [(names_[i], vals[i]) for i in order if vals[i] < threshold]
    uncertain = disagreement > 0.22 or abs(confidence - threshold) < 0.08
    lines = []
    lines.append(f"### Verdict: **{verdict}**  (confidence {confidence:.2f}, spread {disagreement:.2f})")
    if uncertain:
        lines.append("> ⚠️ **Uncertain** — the measures disagree or the score is near the boundary.")
    lines.append(f"\n**Supporting evidence ({len(supporting)} measures >= {threshold}):**")
    for n, v in supporting[:5] or [("(none)", 0)]:
        lines.append(f"- `{n}` = {v:.2f}")
    lines.append(f"\n**Dissenting evidence ({len(dissenting)} measures < {threshold}):**")
    for n, v in dissenting[:5] or [("(none)", 0)]:
        lines.append(f"- `{n}` = {v:.2f}")
    return {"verdict": verdict, "confidence": confidence,
            "disagreement": disagreement, "uncertain": uncertain,
            "scores": scores}, "\n".join(lines)

# explain a Type-2 clone and a hard negative
from clone_pairs import load_pairs
by_id = {p["id"]: p for p in load_pairs()}
for pid in ["java_type2_sum_renamed", "java_negative_sum_vs_max", "python_type4_sum_formula"]:
    p = by_id[pid]
    info, report = explain_pair(p["code_a"], p["code_b"])
    print("=" * 72)
    print(f"PAIR {pid}  (true label = {p['label']}, type = {p['clone_type']})")
    print(report)
    print()

### Reading the explanations

- For the **Type-2** clone, the rename-robust and structural measures dominate the supporting evidence — a human can see *why* it was flagged.
- For the **hard negative**, you will often see a **split** (some measures high, some low) and an **uncertainty warning** — exactly the cases a reviewer should look at.
- For the **Type-4** pair, most lexical measures dissent; this honestly exposes the limit of surface measures.

You can export `info` as JSON, or `report` as Markdown/HTML, to attach machine-readable explanations to each prediction.

## Where to go next

- **Use the full 21 measures:** `from similarity import *` (note: some measures pull in PyTorch/Transformers/Pygments — see the repo's dependency list).
- **Reproduce the paper's ensemble:** `ensembles/bagging.py` (Random Forest) and `ensembles/boosting.py` (XGBoost), plus `utils/calculate_individual_metrics.py` for per-measure tables.
- **Evaluate your *own* detector:** produce a feature/score file in the same `[scores..., label]` JSONL format and reuse `eval_edu.load_feature_file` + `best_threshold_f1`.

### Exercises
1. Replace logistic regression with a decision stump per measure and a majority vote. Does it close the gap to the Random Forest?
2. Add **MCC** alongside F1 in `evaluate()` and re-rank the systems. Does the ranking change under heavy imbalance?
3. Extend `explain_pair` to emit a JSON file per decision and a small HTML report.